# Chicago Crime Analysis
## Requête 2 — Pattern Mining & Analyse Avancée
**Auteure : Ekta**

**Objectif :** Identifier des associations récurrentes entre les caractéristiques 
des crimes (type, moment de la journée, lieu, arrestation) grâce à l'algorithme Apriori.

---

In [15]:
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder
import plotly.graph_objects as go
import plotly.express as px

In [11]:
import plotly.io as pio
pio.renderers.default = "notebook"

### Chargement du dataset

In [16]:
# Fonction pour charger le dataset
# Input : chemin vers le fichier CSV
# Output : DataFrame pandas nettoyé
def load_data(path="../data/chicago_crime.csv"):
    df = pd.read_csv(path)
    df["Date"] = pd.to_datetime(df["Date"], format="mixed")
    return df

df = load_data()
print(f"Dataset chargé : {df.shape[0]} lignes × {df.shape[1]} colonnes")
df.head()

Dataset chargé : 949 lignes × 20 colonnes


,ID,Case Number,Date,Block,IUCR,Primary Type,Description,Location Description,Arrest,Domestic,Beat,Ward,FBI Code,X Coordinate,Y Coordinate,Year,Updated On,Latitude,Longitude,Location
0,14216537,JK278149,2026-06-01 21:58:00,007XX W 31ST ST,0560,ASSAULT,SIMPLE,DRUG STORE,True,False,915,11,08A,1171697.0,1884328.0,2026,2026 Jun 09 04:13:19 PM,"41,83805499","-87,645458414",POINT (-87.645458414 41.83805499)
1,14203870,JK262663,2026-05-20 17:50:00,007XX W 31ST ST,0850,THEFT,ATTEMPT THEFT,SIDEWALK,False,False,915,11,06,1171668.0,1884327.0,2026,2026 May 28 03:42:02 PM,"41,838052883","-87,645564858",POINT (-87.645564858 41.838052883)
2,14178251,JK231486,2026-04-25 22:55:00,007XX W 31ST ST,0860,THEFT,RETAIL THEFT,CONVENIENCE STORE,False,False,915,11,06,1171697.0,1884328.0,2026,2026 May 03 03:43:04 PM,"41,83805499","-87,645458414",POINT (-87.645458414 41.83805499)
3,14134795,JK178608,2026-03-12 15:07:00,007XX W 31ST ST,0560,ASSAULT,SIMPLE,STREET,False,False,915,11,08A,1171697.0,1884328.0,2026,2026 Mar 20 03:43:34 PM,"41,83805499","-87,645458414",POINT (-87.645458414 41.83805499)
4,14114764,JK154429,2026-02-18 18:30:00,007XX W 31ST ST,0820,THEFT,$500 AND UNDER,SMALL RETAIL STORE,False,False,915,11,06,1171668.0,1884327.0,2026,2026 Mar 14 03:41:39 PM,"41,838052883","-87,645564858",POINT (-87.645564858 41.838052883)


## Étape 1 : Discrétisation des variables

Avant d'appliquer l'algorithme Apriori, on doit transformer les variables continues 
ou textuelles en **catégories discrètes** :

| Variable | Transformation |
|---|---|
| `Date` (heure) | matin (6h-12h) / après-midi (12h-18h) / soir (18h-23h) / nuit (0h-6h) |
| `Primary Type` | gardé tel quel (type de crime) |
| `Location Description` | simplifié en grandes catégories (rue, intérieur, transport...) |
| `Arrest` | Arrestation_OUI / Arrestation_NON |
| `Domestic` | Domestique_OUI / Domestique_NON |

Cette étape est indispensable car Apriori travaille uniquement sur des données **booléennes**.

In [4]:
#fonction de discrétisation des variables
# Input : DataFrame brut
# Output : DataFrame avec colonnes discrétisées

def discretiser(df):
    df = df.copy()
    
    # 1. Discrétisation de l'heure en 4 tranches
    heure = df["Date"].dt.hour
    def tranche_horaire(h):
        if 6 <= h < 12:
            return "Matin"
        elif 12 <= h < 18:
            return "Après-midi"
        elif 18 <= h < 23:
            return "Soir"
        else:
            return "Nuit"
    df["Heure"] = heure.apply(tranche_horaire)
    
    # 2. Simplification du lieu (garder les 5 lieux les plus fréquents, regrouper le reste)
    top_lieux = df["Location Description"].value_counts().nlargest(5).index
    df["Lieu"] = df["Location Description"].apply(
        lambda x: x if x in top_lieux else "AUTRE"
    )
    
    # 3. Arrestation en texte lisible
    df["Arrestation"] = df["Arrest"].apply(lambda x: "Arrestation_OUI" if x else "Arrestation_NON")
    
    # 4. Crime domestique
    df["Domestique"] = df["Domestic"].apply(lambda x: "Domestique_OUI" if x else "Domestique_NON")

    df_disc = df[["Primary Type", "Heure", "Lieu", "Arrestation", "Domestique"]]
    
    return df_disc

df_disc = discretiser(df)
print("Discrétisation terminée")
print(df_disc["Heure"].value_counts())
print(df_disc["Lieu"].value_counts())
df_disc.head()

Discrétisation terminée
Heure
Après-midi    353
Soir          259
Nuit          199
Matin         138
Name: count, dtype: int64
Lieu
AUTRE                             337
DRUG STORE                        226
SMALL RETAIL STORE                140
STREET                             83
PARKING LOT/GARAGE(NON.RESID.)     83
RESTAURANT                         80
Name: count, dtype: int64


,Primary Type,Heure,Lieu,Arrestation,Domestique
0,ASSAULT,Soir,DRUG STORE,Arrestation_OUI,Domestique_NON
1,THEFT,Après-midi,AUTRE,Arrestation_NON,Domestique_NON
2,THEFT,Soir,AUTRE,Arrestation_NON,Domestique_NON
3,ASSAULT,Après-midi,STREET,Arrestation_NON,Domestique_NON
4,THEFT,Soir,SMALL RETAIL STORE,Arrestation_NON,Domestique_NON


**Répartition temporelle :**
- L'après-midi (12h-18h) est la tranche horaire la plus criminogène avec 353 incidents (37%)
- La nuit (0h-6h) représente 199 incidents , moins fréquente mais potentiellement plus dangereuse
- Le matin est la période la plus calme avec seulement 138 incidents

**Lieux les plus fréquents :**
- "AUTRE" regroupe tous les lieux rares :337 incidents
- Le DRUG STORE est le lieu le plus identifié avec 226 incidents, ce qui est remarquable 
  pour un seul type de lieu
- La rue (STREET) et les parkings représentent chacun 83 incidents

In [17]:
# Fonction d'encodage en format binaire (one-hot)
# Input : DataFrame discrétisé
# Output : DataFrame binaire prêt pour Apriori

def encoder_transactions(df_disc):
    #transforme chaque ligne en liste d'items
    transactions = df_disc.apply(lambda row: list(row.values), axis=1).tolist()
    
    te = TransactionEncoder()
    te_array = te.fit(transactions).transform(transactions)
    df_encoded = pd.DataFrame(te_array, columns=te.columns_)
    
    return df_encoded

df_encoded = encoder_transactions(df_disc)
print(f"Matrice encodée : {df_encoded.shape[0]} lignes × {df_encoded.shape[1]} colonnes")
df_encoded.head()

Matrice encodée : 949 lignes × 34 colonnes


,ASSAULT,AUTRE,Après-midi,Arrestation_NON,Arrestation_OUI,BATTERY,BURGLARY,CRIMINAL DAMAGE,CRIMINAL TRESPASS,DECEPTIVE PRACTICE,...,PARKING LOT/GARAGE(NON.RESID.),PUBLIC PEACE VIOLATION,RESTAURANT,ROBBERY,SEX OFFENSE,SMALL RETAIL STORE,STREET,Soir,THEFT,WEAPONS VIOLATION
0,True,False,False,False,True,False,False,False,False,False,...,False,False,False,False,False,False,False,True,False,False
1,False,True,True,True,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,True,False
2,False,True,False,True,False,False,False,False,False,False,...,False,False,False,False,False,False,False,True,True,False
3,True,False,True,True,False,False,False,False,False,False,...,False,False,False,False,False,False,True,False,False,False
4,False,False,False,True,False,False,False,False,False,False,...,False,False,False,False,False,True,False,True,True,False


## Étape 2 : Algorithme Apriori & Règles d'association



L'algorithme **Apriori** cherche des combinaisons d'items qui apparaissent souvent ensemble 
dans le dataset. 

On applique l'algorithme Apriori pour extraire les itemsets fréquents, puis on génère les règles d'association avec confidence et lift.
    
- min_support=0.1 signifie qu'on garde les items présents dans au moins 10% des transactions.

On choisit 0.1 car notre dataset est petit (949 lignes).

In [19]:
# Fonction d'extraction des itemsets fréquents et règles d'association
# Input : DataFrame encodé, support minimal
# Output : DataFrame des règles d'association

def appliquer_apriori(df_encoded, min_support=0.1):
    # Extraction des itemsets fréquents
    itemsets = apriori(df_encoded, min_support=min_support, use_colnames=True)
    itemsets = itemsets.sort_values("support", ascending=False)
    
    # Génération des règles d'association
    regles = association_rules(itemsets, metric="lift", min_threshold=1.0)
    regles = regles.sort_values("lift", ascending=False)
    
    print(f"Itemsets fréquents trouvés : {len(itemsets)}")
    print(f"Règles d'association trouvées : {len(regles)}")
    
    return itemsets, regles

itemsets, regles = appliquer_apriori(df_encoded, min_support=0.1)
regles[["antecedents", "consequents", "support", "confidence", "lift"]].head(10)

Itemsets fréquents trouvés : 57
Règles d'association trouvées : 100


,antecedents,consequents,support,confidence,lift
55,"frozenset({Domestique_NON, DRUG STORE})",frozenset({Arrestation_OUI}),0.133825,0.579909,1.667677
58,frozenset({Arrestation_OUI}),"frozenset({Domestique_NON, DRUG STORE})",0.133825,0.384848,1.667677
44,frozenset({DRUG STORE}),frozenset({Arrestation_OUI}),0.136986,0.575221,1.654197
45,frozenset({Arrestation_OUI}),frozenset({DRUG STORE}),0.136986,0.393939,1.654197
59,frozenset({DRUG STORE}),"frozenset({Domestique_NON, Arrestation_OUI})",0.133825,0.561947,1.651045
54,"frozenset({Domestique_NON, Arrestation_OUI})",frozenset({DRUG STORE}),0.133825,0.393189,1.651045
52,frozenset({THEFT}),"frozenset({Domestique_NON, DRUG STORE})",0.136986,0.335917,1.455642
49,"frozenset({Domestique_NON, DRUG STORE})",frozenset({THEFT}),0.136986,0.593607,1.455642
53,frozenset({DRUG STORE}),"frozenset({Domestique_NON, THEFT})",0.136986,0.575221,1.421575
48,"frozenset({Domestique_NON, THEFT})",frozenset({DRUG STORE}),0.136986,0.338542,1.421575


### Résultats de l'algorithme Apriori

L'algorithme a extrait **57 itemsets fréquents** et généré **100 règles d'association**.

### Top règles par lift — Interprétation

**Règle 1 — La plus forte (lift = 1.67) :**
> "Si crime dans un DRUG STORE ET non domestique → Arrestation probable"
- Confidence = 0.58 : dans 58% des cas, cette combinaison mène à une arrestation
- Lift = 1.67 : cette association est 1.67x plus fréquente que par hasard
- Interprétation : les crimes en drug store sont des lieux où la police intervient 
  efficacement, probablement grâce à la présence de caméras et de personnel

**Règle 2 (lift = 1.65) :**
> "Si crime dans un DRUG STORE → Arrestation probable"
- Confirme que le lieu DRUG STORE est fortement associé aux arrestations
- Support = 0.137 : concerne 13.7% de l'ensemble des incidents

**Règle 3 (lift = 1.46) :**
> "Si THEFT dans un DRUG STORE → crime non domestique"
- Les vols en drug store sont systématiquement des crimes non domestiques
- Ce résultat est logique : le vol à l'étalage est un crime public, pas familial

### Conclusion générale
Le pattern mining révèle que le **DRUG STORE** est un lieu central dans la criminalité 
de Bridgeport, fortement associé aux arrestations. Les crimes non domestiques commis 
dans ces lieux ont une forte probabilité de mener à une interpellation, 
ce qui suggère une présence policière ou une surveillance accrue dans ces commerces.

## Impact du support minimal sur le nombre de patterns

On teste différentes valeurs de support minimal σ et on trace le nombre d'itemsets fréquents trouvés pour chaque valeur.
Ceci nous permet de choisir un bon compromis entre quantité et pertinence des règles.

In [22]:
# Fonction pour analyser l'impact du support minimal sur le nombre de patterns
# Input : DataFrame encodé
# Output : graphique Plotly

def graphique_support(df_encoded):
    supports = [0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4]
    nb_itemsets = []
    
    for s in supports:
        items = apriori(df_encoded, min_support=s, use_colnames=True)
        nb_itemsets.append(len(items))
    
    fig = px.line(
        x=supports,
        y=nb_itemsets,
        markers=True,
        title="Impact du support minimal σ sur le nombre d'itemsets fréquents",
        labels={"x": "Support minimal σ", "y": "Nombre d'itemsets fréquents"}
    )
    fig.add_vline(
        x=0.1, 
        line_dash="dash", 
        line_color="red",
        annotation_text="σ choisi = 0.1"
    )
    fig.show()
    return fig

fig_support = graphique_support(df_encoded)

Le graphique montre une chute brutale entre σ=0.05 (140 itemsets) et σ=0.1 (57 itemsets).

Au-delà de σ=0.1, la courbe s'aplatit progressivement jusqu'à seulement 5 itemsets à σ=0.4.

Nous avons choisi σ=0.1 car c'est le meilleur compromis :
- Assez bas pour capturer des associations intéressantes (57 itemsets)
- Assez élevé pour éliminer les coïncidences rares non significatives
- Correspond à des items présents dans au moins ~95 transactions sur 949

## Étape 3 — Visualisation Sankey

Le **diagramme de Sankey** permet de visualiser les règles d'association :
Dans ce graph les antécédents sont à gauche, les conséquents à droite.
L'épaisseur des liens représente la confidence, la couleur représente le lift.
On garde les top_n règles triées par lift pour ne garder que les plus intéressantes.

In [23]:
# Fonction de visualisation Sankey des règles d'association
# Input : DataFrame des règles
# Output : graphique Sankey Plotly

def sankey_regles(regles, top_n=15):
    # Garder les meilleures règles
    top_regles = regles.nlargest(top_n, "lift").copy()
    
    # Convertir les frozensets en strings lisibles
    top_regles["ant_str"] = top_regles["antecedents"].apply(lambda x: ", ".join(list(x)))
    top_regles["cons_str"] = top_regles["consequents"].apply(lambda x: ", ".join(list(x)))
    
    # Créer les noeuds uniques
    all_labels = list(set(top_regles["ant_str"].tolist() + top_regles["cons_str"].tolist()))
    label_idx = {label: i for i, label in enumerate(all_labels)}
    
    # Créer les liens
    sources = [label_idx[a] for a in top_regles["ant_str"]]
    targets = [label_idx[c] for c in top_regles["cons_str"]]
    values = top_regles["confidence"].tolist()
    lifts = top_regles["lift"].tolist()
    
    # Couleurs selon le lift
    colors = [f"rgba(255, {max(0, int(255 - l*30))}, 0, 0.5)" for l in lifts]
    
    fig = go.Figure(go.Sankey(
        node=dict(
            pad=15,
            thickness=20,
            line=dict(color="black", width=0.5),
            label=all_labels,
            color="lightblue"
        ),
        link=dict(
            source=sources,
            target=targets,
            value=values,
            color=colors
        )
    ))
    
    fig.update_layout(
        title_text="Règles d'association :Diagramme de Sankey (épaisseur = confidence, couleur = lift)",
        font_size=12,
        height=600
    )
    fig.show()
    return fig

fig_sankey = sankey_regles(regles, top_n=15)

Chaque flux représente une règle d'association :
- **Gauche** = antécédent (condition SI)
- **Droite** = conséquent (conclusion ALORS)
- **Épaisseur du lien** = confidence (plus c'est épais, plus la règle est fiable)
- **Couleur** = lift (plus c'est chaud/orange, plus l'association est forte)

Le diagramme confirme visuellement la dominance du **DRUG STORE** comme nœud central, 
concentrant les flux les plus épais vers **Arrestation_OUI**.

In [25]:
# Bloc principal =>  appelle toutes les fonctions dans l'ordre

if __name__ == "__main__":
    # Étape 1 : Chargement
    df = load_data("../data/chicago_crime.csv")
    
    # Étape 2 : Discrétisation
    df_disc = discretiser(df)
    
    # Étape 3 : Encodage
    df_encoded = encoder_transactions(df_disc)
    
    # Étape 4 : Apriori
    itemsets, regles = appliquer_apriori(df_encoded, min_support=0.1)
    
    # Étape 5 : Visualisations
    fig_support = graphique_support(df_encoded)
    fig_sankey = sankey_regles(regles, top_n=15)
    
    print("Pattern Mining terminé !")

Itemsets fréquents trouvés : 57
Règles d'association trouvées : 100


Pattern Mining terminé !


## Conclusion — Pattern Mining

Cette analyse par règles d'association a permis d'identifier des patterns récurrents 
dans la criminalité du quartier de Bridgeport à Chicago.

Les principaux enseignements sont :
- Le DRUG STORE est le lieu le plus criminogène et le plus associé aux arrestations
- Les crimes commis **l'après-midi** sont les plus fréquents
- Les crimes **non domestiques** en drug store mènent très souvent à une arrestation (lift > 1.6)

Ces patterns constituent une base solide pour anticiper les zones et situations à risque,
et peuvent aider à mieux orienter les ressources policières sur le terrain.
